# 05 — Retrieval Lookup + Final Ensemble
**Roll No:** 23f3004491 | Model 5 of 5 | Milestones 3 & 5

The component that decided the score. After stripping wrappers, ~98% of test questions duplicate
a training question. A three-tier lookup exploits this and becomes the primary predictor, hedged
by the cross-encoder. This is the pipeline that produced the final leaderboard score of 0.75685.

In [ ]:
import warnings, re
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

BASE = "/kaggle/input/competitions/smart-mcq-solver-challenge"
train = pd.read_csv(f"{BASE}/train.csv")
test  = pd.read_csv(f"{BASE}/test.csv")
OPTIONS = ["A", "B", "C", "D", "E"]

def average_precision_at_3(true_label, predicted_labels):
    for rank, pred in enumerate(predicted_labels[:3]):
        if pred == true_label:
            return 1.0 / (rank + 1)
    return 0.0

def mean_average_precision_at_3(true_labels, predicted_lists):
    return float(np.mean([average_precision_at_3(t, p)
                          for t, p in zip(true_labels, predicted_lists)]))

START_WRAPPERS = ["Pick the best possible answer:", "Select the most accurate option:",
                  "Determine the correct option:", "Identify the correct statement:",
                  "Choose the correct answer:"]

def normalize_core(prompt):
    p = str(prompt).strip()
    for s in START_WRAPPERS:
        if p.startswith(s):
            p = p[len(s):].strip()
    if "?" in p:
        p = p[:p.rfind("?") + 1]
    return re.sub(r"\s+", " ", p).lower().strip()

train["core"] = train["prompt"].apply(normalize_core)
test["core"]  = test["prompt"].apply(normalize_core)

np.random.seed(42)
cores = train["core"].unique().copy()
np.random.shuffle(cores)
val_cores = set(cores[:200])
valid_df = train[train["core"].isin(val_cores)].drop_duplicates("core").reset_index(drop=True)
print("train:", train.shape, "| validation questions:", len(valid_df))

In [ ]:
!pip install -q sentence-transformers

In [ ]:
import difflib
from collections import Counter, defaultdict

# ---- Tier 0: option text is a known-correct answer somewhere in train ----
train["answer_text"] = train.apply(lambda r: str(r[r["answer"]]).strip(), axis=1)
KNOWN_ANSWER_TEXTS = set(train["answer_text"].str.lower())

# ---- Tier 1: exact option-set -> majority answer ----
def option_signature(row):
    return "||".join(sorted(str(row[o]).strip().lower() for o in OPTIONS))
train["osig"] = train.apply(option_signature, axis=1)
_osig = defaultdict(list)
for sig, ans in zip(train["osig"], train["answer"]):
    _osig[sig].append(ans)
osig_letter = {s: Counter(v).most_common(1)[0][0] for s, v in _osig.items()}

# ---- Tier 2: core question -> majority answer text ----
_core = defaultdict(list)
for core, atext in zip(train["core"], train["answer_text"]):
    _core[core].append(atext)
core_answer = {c: Counter(v).most_common(1)[0][0] for c, v in _core.items()}

def lookup_letter(row):
    matches = [o for o in OPTIONS if str(row[o]).strip().lower() in KNOWN_ANSWER_TEXTS]
    if len(matches) == 1:
        return matches[0]
    sig = option_signature(row)
    if sig in osig_letter:
        return osig_letter[sig]
    ans = core_answer.get(row["core"])
    if ans is not None:
        for o in OPTIONS:
            if str(row[o]).strip() == ans:
                return o
        sims = sorted(((difflib.SequenceMatcher(None, str(row[o]).strip().lower(),
                                                ans.lower()).ratio(), o) for o in OPTIONS),
                      reverse=True)
        if sims[0][0] > 0.80:
            return sims[0][1]
    return None

In [ ]:
from sentence_transformers import CrossEncoder
ce_model = CrossEncoder("cross-encoder/nli-deberta-v3-small")

def cross_encoder_scores(df):
    pairs = [(row["prompt"], row[o]) for _, row in df.iterrows() for o in OPTIONS]
    logits = ce_model.predict(pairs, batch_size=32, show_progress_bar=False)
    entail = logits[:, 1] if logits.ndim == 2 else logits
    return entail.reshape(len(df), 5)

FREQ_ORDER = train["answer"].value_counts().index.tolist()

def length_freq_scores(df):
    length = df[OPTIONS].astype(str).apply(lambda c: c.str.len()).values.astype(float)
    freq = np.array([[(5 - FREQ_ORDER.index(o)) for o in OPTIONS]] * len(df), dtype=float)
    def nrm(S):
        lo = S.min(1, keepdims=True); hi = S.max(1, keepdims=True)
        return (S - lo) / (hi - lo + 1e-9)
    return nrm(length) + 0.3 * nrm(freq)

In [ ]:
def build_predictions(df):
    ce = cross_encoder_scores(df)
    lf = length_freq_scores(df)
    def nrm(S):
        lo = S.min(1, keepdims=True); hi = S.max(1, keepdims=True)
        return (S - lo) / (hi - lo + 1e-9)
    ce = nrm(ce)
    preds = []
    for i in range(len(df)):
        ce_rank = [OPTIONS[j] for j in np.argsort(ce[i])[::-1]]
        lf_rank = [OPTIONS[j] for j in np.argsort(lf[i])[::-1]]
        la = lookup_letter(df.iloc[i])
        if la is not None:
            ranked = [la]
            for o in ce_rank:
                if o not in ranked: ranked.append(o); break
            for o in lf_rank:
                if o not in ranked: ranked.append(o); break
            preds.append(ranked[:3])
        else:
            preds.append((ce_rank + [o for o in lf_rank if o not in ce_rank])[:3])
    return preds

preds = build_predictions(valid_df)
score = mean_average_precision_at_3(valid_df["answer"].tolist(), preds)
print(f"Retrieval + ensemble validation MAP@3: {score:.4f}")

In [ ]:
# ---- generate the Kaggle submission ----
test_preds = build_predictions(test)
submission = pd.DataFrame({"ID": test["id"],
                           "Prediction": [" ".join(p[:3]) for p in test_preds]})
submission.to_csv("submission.csv", index=False)
assert list(submission.columns) == ["ID", "Prediction"]
assert submission["Prediction"].str.split().str.len().eq(3).all()
print("submission.csv ready —", len(submission), "rows")

## Observation
The three-tier retrieval lookup, hedged by the cross-encoder and a length-frequency signal,
reaches the highest validation score and produced the final public leaderboard result of
**0.75685**. The lookup is the main driver; the models only help on the small set of novel or
altered questions.